# 📓 Notebook 04 — Deep Learning: MLP Neural Network
## Student Placement Prediction System
### Deep Learning Extension — All Members

This notebook demonstrates:
- Multi-Layer Perceptron (MLP) theory and architecture
- Training a neural network for binary classification
- Visualising the loss curve (training progress)
- Comparing shallow vs. deep architectures
- L2 regularization (alpha) sweep for MLPs
- GridSearchCV tuning
- Comparing MLP with LR, DT, RF on the same test set

---

## 🧠 What is a Neural Network?

A **Multi-Layer Perceptron** is a stack of fully connected layers:

```
Input Features → [Hidden Layer 1: 128 neurons] → [Hidden Layer 2: 64 neurons] → Output
                      ReLU activation                   ReLU activation          Sigmoid
```

Each neuron computes: **z = W·x + b**, then **a = f(z)**

- **W** (weights), **b** (bias): learnable parameters  
- **f** (activation): introduces non-linearity  
- **Loss** (Binary Cross-Entropy): measures prediction error  
- **Backpropagation**: computes gradients of loss w.r.t. every weight  
- **Adam optimizer**: updates weights adaptively using momentum  

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
plt.rcParams['figure.dpi'] = 100

from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)

from data_preprocessing import run_full_preprocessing

(X_train, X_test, y_train, y_test,
 preprocessor, df_clean,
 numeric_features, categorical_features, target_col) = run_full_preprocessing()

print(f'Train: {X_train.shape} | Test: {X_test.shape}')
print('Data ready ✓')

## Step 1: Baseline MLP Neural Network

**Architecture**: 2 hidden layers → 128 neurons → 64 neurons → 1 output  
**Activation**: ReLU (hidden), Logistic/Sigmoid (output)  
**Optimizer**: Adam  
**Early Stopping**: Yes (stops when validation loss plateaus for 15 iterations)

In [ ]:
mlp_baseline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(
        hidden_layer_sizes=(128, 64),  # 2 hidden layers
        activation='relu',             # ReLU: max(0, z)
        solver='adam',                 # Adaptive Moment Estimation
        alpha=0.0001,                  # L2 regularization
        learning_rate='adaptive',
        max_iter=500,
        early_stopping=True,           # stops when val loss plateaus
        validation_fraction=0.1,
        n_iter_no_change=15,
        random_state=42,
        verbose=False,
    ))
])

mlp_baseline.fit(X_train, y_train)

model = mlp_baseline.named_steps['model']
print(f'Iterations (epochs): {model.n_iter_}')
print(f'Final training loss: {model.loss_:.4f}')
print(f'Architecture: {model.hidden_layer_sizes}')
print(f'\nTest set report:')
print(classification_report(y_test, mlp_baseline.predict(X_test),
                             target_names=['Not Placed', 'Placed']))

## Step 2: Training Loss Curve

**Interpretation**:  
- Decreasing curve = model is learning  
- Flattening at the end = early stopping fired (convergence)  
- If validation diverges from training = overfitting

In [ ]:
model = mlp_baseline.named_steps['model']

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(model.loss_curve_, color='#3498DB', linewidth=2, label='Training Loss')

if hasattr(model, 'validation_scores_') and model.validation_scores_ is not None:
    # validation_scores_ is accuracy → convert to error
    ax.plot([1 - v for v in model.validation_scores_],
            color='#E74C3C', linewidth=2, linestyle='--',
            label='Validation (1 - Accuracy)')

ax.set_xlabel('Iteration (Epoch)')
ax.set_ylabel('Loss (Cross-Entropy)')
ax.set_title('MLP Neural Network — Training Loss Curve\n(Early stopping fires when loss plateaus)', fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print(f'Training converged in {model.n_iter_} iterations.')

## Step 3: Architecture Comparison — Shallow vs Deep

| Architecture | Description |
|---|---|
| (64,) | 1 hidden layer, 64 neurons — simple |
| (128, 64) | 2 hidden layers — recommended for tabular data |
| (256, 128, 64) | 3 hidden layers — more expressive, higher overfitting risk |

In [ ]:
architectures = {
    'Shallow (64)'       : (64,),
    'Standard (128, 64)' : (128, 64),
    'Deep (256, 128, 64)': (256, 128, 64),
}

arch_results = {}
for name, arch in architectures.items():
    pipe = Pipeline([
        ('pp', preprocessor),
        ('m', MLPClassifier(
            hidden_layer_sizes=arch, activation='relu', solver='adam',
            alpha=0.0001, max_iter=500, early_stopping=True,
            validation_fraction=0.1, n_iter_no_change=15,
            random_state=42, verbose=False
        ))
    ])
    pipe.fit(X_train, y_train)
    train_f1 = f1_score(y_train, pipe.predict(X_train), zero_division=0)
    test_f1  = f1_score(y_test,  pipe.predict(X_test),  zero_division=0)
    arch_results[name] = {'train_f1': train_f1, 'test_f1': test_f1}
    print(f'{name:28s}  Train F1: {train_f1:.4f}  |  Test F1: {test_f1:.4f}')

# Plot
names = list(arch_results.keys())
x = np.arange(len(names))
w = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - w/2, [arch_results[n]['train_f1'] for n in names], w,
       color='#3498DB', label='Train F1', alpha=0.85)
ax.bar(x + w/2, [arch_results[n]['test_f1']  for n in names], w,
       color='#E74C3C', label='Test F1', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=10, ha='right')
ax.set_ylim(0, 1.1)
ax.set_ylabel('F1-Score')
ax.set_title('MLP Architecture Comparison: Train vs Test F1', fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

print('\nA large gap between Train F1 and Test F1 indicates overfitting.')
print('Deeper models may overfit more on synthetic tabular data.')

## Step 4: L2 Regularization (Alpha) Sweep

**alpha** controls the L2 weight penalty in MLP (same concept as λ in LR regularization).  
- Small alpha → complex model → risk of overfitting  
- Large alpha → simpler model → risk of underfitting

In [ ]:
alphas = [1e-5, 1e-4, 1e-3, 1e-2, 0.1, 1.0]
train_scores, test_scores = [], []

for a in alphas:
    pipe = Pipeline([
        ('pp', preprocessor),
        ('m', MLPClassifier(
            hidden_layer_sizes=(128, 64), alpha=a,
            activation='relu', solver='adam',
            max_iter=500, early_stopping=True,
            validation_fraction=0.1, n_iter_no_change=15,
            random_state=42, verbose=False
        ))
    ])
    pipe.fit(X_train, y_train)
    train_scores.append(f1_score(y_train, pipe.predict(X_train), zero_division=0))
    test_scores.append(f1_score(y_test,   pipe.predict(X_test),  zero_division=0))
    print(f'alpha={a:.0e}  train F1={train_scores[-1]:.4f}  test F1={test_scores[-1]:.4f}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(alphas, train_scores, 'o-', color='#3498DB', linewidth=2, label='Train F1')
ax.plot(alphas, test_scores,  's--', color='#E74C3C', linewidth=2, label='Test F1')
ax.set_xscale('log')
ax.set_xlabel('Alpha (L2 Regularization Strength)')
ax.set_ylabel('F1-Score')
ax.set_title('MLP Neural Network: L2 Regularization (Alpha) Sweep', fontweight='bold')
ax.legend()
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

best_alpha = alphas[np.argmax(test_scores)]
print(f'\nBest alpha by test F1: {best_alpha}')

## Step 5: GridSearchCV Tuning

In [ ]:
param_grid = {
    'model__hidden_layer_sizes': [(64,), (128, 64), (128, 64, 32)],
    'model__alpha'             : [1e-4, 1e-3, 1e-2],
    'model__learning_rate_init': [0.001, 0.005],
}

pipe_gs = Pipeline([
    ('preprocessor', preprocessor),
    ('model', MLPClassifier(
        activation='relu', solver='adam',
        max_iter=500, early_stopping=True,
        validation_fraction=0.1, n_iter_no_change=15,
        random_state=42, verbose=False
    ))
])

print('Running MLP GridSearchCV (5-fold, F1 scoring)...')
gs = GridSearchCV(pipe_gs, param_grid, cv=5, scoring='f1', n_jobs=-1, verbose=1, refit=True)
gs.fit(X_train, y_train)
mlp_tuned = gs.best_estimator_

print(f'\nBest params : {gs.best_params_}')
print(f'Best CV F1  : {gs.best_score_:.4f}')
print(f'Test F1     : {f1_score(y_test, mlp_tuned.predict(X_test)):.4f}')

## Step 6: Final Comparison — MLP vs Classical Models

Compare MLP (Tuned) against Logistic Regression, Decision Tree, and Random Forest.

In [ ]:
import joblib

MODELS_DIR = os.path.join('..', 'models')

def load_if_exists(name):
    path = os.path.join(MODELS_DIR, f'{name}.pkl')
    return joblib.load(path) if os.path.exists(path) else None

all_models = {
    'LR Tuned'  : load_if_exists('lr_tuned'),
    'DT Tuned'  : load_if_exists('dt_tuned'),
    'RF Tuned'  : load_if_exists('rf_tuned'),
    'MLP Baseline': mlp_baseline,
    'MLP Tuned' : mlp_tuned,
}

rows = []
for name, pipe in all_models.items():
    if pipe is None:
        print(f'  Skipping {name} — not found. Run run_pipeline.py first.')
        continue
    y_pred = pipe.predict(X_test)
    try:
        auc = roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])
    except:
        auc = None
    rows.append({
        'Model'    : name,
        'Accuracy' : round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred, zero_division=0), 4),
        'Recall'   : round(recall_score(y_test, y_pred, zero_division=0), 4),
        'F1-Score' : round(f1_score(y_test, y_pred, zero_division=0), 4),
        'ROC-AUC'  : round(auc, 4) if auc else 'N/A',
    })

cmp_df = pd.DataFrame(rows)
print('\n📊 COMPLETE MODEL COMPARISON (Test Set):')
print('=' * 65)
print(cmp_df.to_string(index=False))

In [ ]:
# ROC Curves — all models including MLP
fig, ax = plt.subplots(figsize=(8, 6))
colours = ['#9B59B6', '#E74C3C', '#2ECC71', '#3498DB', '#F39C12']
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random (AUC=0.50)')

for i, (name, pipe) in enumerate(all_models.items()):
    if pipe is None:
        continue
    try:
        y_prob = pipe.predict_proba(X_test)[:, 1]
        fpr, tpr, _ = roc_curve(y_test, y_prob)
        auc = roc_auc_score(y_test, y_prob)
        ax.plot(fpr, tpr, color=colours[i % len(colours)],
                linewidth=2.5 if 'MLP' in name else 1.5,
                linestyle='-' if 'MLP' in name else '--',
                label=f'{name} (AUC={auc:.3f})')
    except:
        pass

ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate (Recall)')
ax.set_title('ROC Curves: MLP vs Classical Models', fontweight='bold')
ax.legend(loc='lower right', fontsize=9)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()
print('Solid lines = MLP variants | Dashed = classical models')

## Summary: Neural Network vs Classical Models

| Aspect | Logistic Regression | Decision Tree | Random Forest | MLP Neural Network |
|--------|--------------------|----|----|---------|
| Interpretability | High | Medium | Low | Very Low |
| Handles non-linearity | ❌ (linear) | ✅ | ✅ | ✅ (best) |
| Needs feature scaling | ✅ | ❌ | ❌ | ✅ |
| Overfitting risk | Low | High | Medium | High |
| Training speed | Fast | Fast | Moderate | Slower |
| Hyperparameters | Few | Moderate | Many | Many |
| Feature importance | Coefficients | Node impurity | MDI | ❌ Not directly |

**For this B.Tech project**: MLP adds a modern deep learning component to the comparison, demonstrating how neural networks fit into the ML landscape alongside classical models.